In [4]:
import importlib
import sys
from pathlib import Path
import pandas as pd

# Set up home path and working repo dir 
HOME = Path.home()
repo_dir = HOME / "OneDrive/git-repos/LCBP-interannual-EMMAs"

if str(repo_dir) not in sys.path:
    sys.path.append(str(repo_dir))

# Import EMMA module
import EMMA.event_emma as em  
import EMMA.event_pca_error as ep
import EMMA.uncertainty_propagation_t as up
importlib.reload(ep)
importlib.reload(em)
importlib.reload(up)

# Define data directory using relative path logic
data_dir = repo_dir / "Data/GrabSample_data"

# Define output directory using relative path logic
output_dir = repo_dir / "Output/EMMA-uncertainty"

# Load the full RI18 through RI25 dataset
# see join @ https://github.com/MeganEDuffy/LCBP-interannual-EMMAs/blob/main/Notebooks/Combining-all-years-RI.ipynb
#df = pd.read_csv(data_dir / "RI18-25-joined.csv")

# Load just the RI23 dataset
df = pd.read_csv(data_dir / "RI23-IC-ICP-isotope-toc-joined.csv")

####################
# Hungerford RI23 events #
####################

Hungerford_tracers = ['Ca_mg_L', 'Cu_mg_L', 'Cl_mg_L', 'K_mg_L', 'Mg_mg_L', 'dD', 'd18O', 'Na_mg_L']

# 1. Run EMMA function
(
    hungerford_febros_fractions_df,
    hungerford_febros_scaler,
    hungerford_febros_pca,
    hungerford_febros_endmembers_df,
) = em.run_emma_event(
    data=df,
    site="Hungerford",
    start_date="2023-02-08 00:00:00",
    end_date="2023-02-12 00:00:00",
    endmember_ids=[
                   "RI23-1001", # Baseflow 02/09/2023
                   "RI23-5003", # SWLD 02/15/2023
                   "RI23-5002" #  SML 02/15/2023
                    ], 
)

# 2. Extract raw endmember rows matching these IDs (to calculate endmember SDs)
em_raw_subset = df[df["Sample ID"].isin(["RI23-1001", "RI23-5003", "RI23-5002"])]

# 3. Define streamwater samples for the event window
stream_event_df = df[
    (df["Site"] == "Hungerford")
    & (df["Type"].isin(["Grab", "Grab/Isco", "Baseflow", "Isco"]))
    & (pd.to_datetime(df["Date"] + " " + df["Time"]) >= pd.to_datetime("2023-02-08 00:00:00"))
    & (pd.to_datetime(df["Date"] + " " + df["Time"]) <= pd.to_datetime("2023-02-12 00:00:00"))
].copy()
stream_event_df["Datetime"] = pd.to_datetime(
    stream_event_df["Date"] + " " + stream_event_df["Time"]
)

# 4. Define tracer analytical uncertainties 
analytical_sd = {
    "Ca_mg_L": 0.05,  # mg/L
    "Cl_mg_L": 0.15,  # mg/L
    "Si_mg_L": 0.10,  # mg/L
    "K_mg_L": 0.10,  # mg/L
    "Cu_mg_L": 0.10,  # mg/L
    "Mg_mg_L": 0.02,  # mg/L
    "dD": 0.8,  # per mil (stable isotope precision)
    "d18O": 0.08,  # per mil
    "Na_mg_L": 0.05,  # mg/L
}

# 5. Calculate Genereux Uncertainties
uncertainty_df = up.propagate_genereux_uncertainty(
    stream_df=stream_event_df,
    em_grouped=hungerford_febros_endmembers_df,
    em_raw=em_raw_subset,
    tracers=Hungerford_tracers,
    analytical_sd=analytical_sd,
    confidence_level=0.95
)

# 6. Merge fractions and their calculated uncertainties for a complete table
results_with_error = pd.merge(
    hungerford_febros_fractions_df, uncertainty_df, on=["Sample ID", "Datetime"]
)

# Identify which uncertainty columns were actually generated
uncertainty_cols = [
    col for col in results_with_error.columns if "Uncertainty_1sig" in col
]
# Extract the base fraction names (e.g., "Groundwater", "Snowmelt lysimeter")
fraction_cols = [col.replace("_Uncertainty_1sig", "") for col in uncertainty_cols]

# Combine them in alternating order: [Fraction_1, Uncertainty_1, Fraction_2, Uncertainty_2...]
display_cols = ["Sample ID", "Datetime"]
for frac, unc in zip(fraction_cols, uncertainty_cols):
    display_cols.extend([frac, unc])

# Print the head of the dynamically built column list
print(results_with_error[display_cols].head())
results_with_error.head(20)

   Sample ID            Datetime
0  RI23-1005 2023-02-10 10:00:00
1  RI23-1007 2023-02-10 16:00:00
2  RI23-1008 2023-02-11 12:30:00
3  RI23-1001 2023-02-09 13:40:00
4  RI23-1002 2023-02-09 16:00:00


,Sample ID,Datetime,Site,Baseflow,Snowmelt lysimeter,Soil water lysimeter,Sum_Fractions,Baseflow_Uncertainty_95sig,Snowmelt lysimeter_Uncertainty_95sig,Soil water lysimeter_Uncertainty_95sig
0,RI23-1005,2023-02-10 10:00:00,Hungerford,0.242656,0.000000e+00,7.573443e-01,1.0,0.857223,8.572230e-01,1.067475e-07
1,RI23-1007,2023-02-10 16:00:00,Hungerford,0.485255,1.999106e-15,5.147447e-01,1.0,0.530474,7.231808e+00,7.081121e+00
2,RI23-1008,2023-02-11 12:30:00,Hungerford,1.000000,0.000000e+00,0.000000e+00,1.0,0.606384,5.427093e+00,5.290130e+00
3,RI23-1001,2023-02-09 13:40:00,Hungerford,1.000000,0.000000e+00,4.996004e-16,1.0,0.788248,4.013553e-01,6.783976e-01
4,RI23-1002,2023-02-09 16:00:00,Hungerford,0.975683,2.431747e-02,0.000000e+00,1.0,0.000191,1.002388e-04,9.071857e-05
5,RI23-1003,2023-02-09 22:00:00,Hungerford,NaN,NaN,NaN,0.0,1.137858,8.537457e-10,1.137858e+00


In [8]:
import importlib
import sys
from pathlib import Path
import pandas as pd

# Set up home path and working repo dir 
HOME = Path.home()
repo_dir = HOME / "OneDrive/git-repos/LCBP-interannual-EMMAs"

if str(repo_dir) not in sys.path:
    sys.path.append(str(repo_dir))

# Import EMMA module
import EMMA.event_emma as em  
import EMMA.event_pca_error as ep
import EMMA.uncertainty_propagation_t as up
importlib.reload(ep)
importlib.reload(em)
importlib.reload(up)

# Define data directory using relative path logic
data_dir = repo_dir / "Data/GrabSample_data"

# Define output directory using relative path logic
output_dir = repo_dir / "Output/EMMA-uncertainty"

# Load the full RI25 dataset
df = pd.read_csv(data_dir / "RI23-IC-ICP-isotope-toc-joined.csv")

####################
# Hungerford RI23 events #
####################

Hungerford_tracers = ['Ca_mg_L', 'Cu_mg_L', 'Cl_mg_L', 'K_mg_L', 'Mg_mg_L', 'dD', 'd18O', 'Na_mg_L']

# 1. Run EMMA function
(
    hungerford_martherm_fractions_df,
    hungerford_martherm_scaler,
    hungerford_martherm_pca,
    hungerford_martherm_endmembers_df,
) = em.run_emma_event(
    data=df,
    site="Hungerford",
    start_date="2023-03-21 00:00:00",
    end_date="2023-03-26 00:00:00",
    endmember_ids=[
                   #"RI23-1001", # Baseflow 02/09/2023
                   "RI23-1035", # 03/22/2023 Baseflow
                   #"RI23-5008", # Mark's well 2023
                   #"RI20-HB-DGW-Mark-1" # Mark's well 2020
                   #"RI22-0858", # Mark's well 03/17/2022
                   #"RI22-0864",
                   #"RI23-1085", #Baseflow 03/31/2023 removed for now
                   "RI23-5007", # SWLD
                   "RI23-1061"
                    ], 
)

# 2. Extract raw endmember rows matching these IDs (to calculate endmember SDs)
em_raw_subset = df[df["Sample ID"].isin([["RI23-1035", "RI23-5007", "RI23-1061"]])]

# 3. Define streamwater samples for the event window
stream_event_df = df[
    (df["Site"] == "Hungerford")
    & (df["Type"].isin(["Grab", "Grab/Isco", "Baseflow", "Isco"]))
    & (pd.to_datetime(df["Date"] + " " + df["Time"]) >= pd.to_datetime("2023-03-21 00:00:00"))
    & (pd.to_datetime(df["Date"] + " " + df["Time"]) <= pd.to_datetime("2023-03-26 00:00:00"))
].copy()
stream_event_df["Datetime"] = pd.to_datetime(
    stream_event_df["Date"] + " " + stream_event_df["Time"]
)

# 4. Define tracer analytical uncertainties (Optional: edit these to match lab detection limits)
analytical_sd = {
    "Ca_mg_L": 0.05,  # mg/L
    "Cl_mg_L": 0.15,  # mg/L
    "Si_mg_L": 0.10,  # mg/L
    "K_mg_L": 0.10,  # mg/L
    "Cu_mg_L": 0.10,  # mg/L
    "Mg_mg_L": 0.02,  # mg/L
    "dD": 0.8,  # per mil (stable isotope precision)
    "d18O": 0.08,  # per mil
    "Na_mg_L": 0.05,  # mg/L
}

# 5. Calculate Genereux Uncertainties
uncertainty_df = up.propagate_genereux_uncertainty(
    stream_df=stream_event_df,
    em_grouped=hungerford_martherm_endmembers_df,
    em_raw=em_raw_subset,
    tracers=Hungerford_tracers,
    analytical_sd=analytical_sd,
    confidence_level=0.70
)

# 6. Merge fractions and their calculated uncertainties for a complete table
results_with_error = pd.merge(
    hungerford_martherm_fractions_df, uncertainty_df, on=["Sample ID", "Datetime"]
)

# Identify which uncertainty columns were actually generated
uncertainty_cols = [
    col for col in results_with_error.columns if "Uncertainty_1sig" in col
]
# Extract the base fraction names (e.g., "Groundwater", "Snowmelt lysimeter")
fraction_cols = [col.replace("_Uncertainty_1sig", "") for col in uncertainty_cols]

# Combine them in alternating order: [Fraction_1, Uncertainty_1, Fraction_2, Uncertainty_2...]
display_cols = ["Sample ID", "Datetime"]
for frac, unc in zip(fraction_cols, uncertainty_cols):
    display_cols.extend([frac, unc])

# Print the head of the dynamically built column list
print(results_with_error[display_cols].head())
results_with_error.head(20)

   Sample ID            Datetime
0  RI23-1035 2023-03-22 15:00:00
1  RI23-1036 2023-03-22 18:00:00
2  RI23-1037 2023-03-23 00:00:00
3  RI23-1038 2023-03-23 06:00:00
4  RI23-1042 2023-03-23 12:00:00


,Sample ID,Datetime,Site,Baseflow,Snowmelt lysimeter,Soil water lysimeter,Sum_Fractions,Baseflow_Uncertainty_70sig,Snowmelt lysimeter_Uncertainty_70sig,Soil water lysimeter_Uncertainty_70sig
0,RI23-1035,2023-03-22 15:00:00,Hungerford,1.000000,7.160939e-15,8.160139e-15,1.0,0.104609,5.564014e-02,8.867601e-02
1,RI23-1036,2023-03-22 18:00:00,Hungerford,0.930323,6.967694e-02,0.000000e+00,1.0,0.171445,5.031091e-07,1.714448e-01
2,RI23-1037,2023-03-23 00:00:00,Hungerford,0.666576,1.043686e-14,3.334245e-01,1.0,0.105074,1.050736e-01,8.475463e-09
3,RI23-1038,2023-03-23 06:00:00,Hungerford,0.676390,6.398332e-14,3.236097e-01,1.0,0.171272,3.772185e-01,4.996686e-01
4,RI23-1042,2023-03-23 12:00:00,Hungerford,0.730678,2.337686e-02,2.459453e-01,1.0,0.178549,4.027303e-01,5.270862e-01
5,RI23-1043,2023-03-23 18:00:00,Hungerford,0.663286,2.845388e-01,5.217558e-02,1.0,0.140518,3.265893e-01,4.297735e-01
6,RI23-1044,2023-03-24 00:00:00,Hungerford,0.395734,6.042662e-01,0.000000e+00,1.0,0.104083,2.374443e-01,3.183208e-01
7,RI23-1045,2023-03-24 06:00:00,Hungerford,0.476257,4.784052e-01,4.533788e-02,1.0,0.121785,2.973724e-01,4.015115e-01
8,RI23-1046,2023-03-24 12:00:00,Hungerford,0.470354,3.838700e-01,1.457761e-01,1.0,0.121232,2.942049e-01,3.935565e-01
9,RI23-1047,2023-03-24 15:45:00,Hungerford,0.599256,3.512834e-01,4.946011e-02,1.0,0.113626,2.627704e-01,3.506090e-01


In [9]:
import importlib
import sys
from pathlib import Path
import pandas as pd

# Set up home path and working repo dir 
HOME = Path.home()
repo_dir = HOME / "OneDrive/git-repos/LCBP-interannual-EMMAs"

if str(repo_dir) not in sys.path:
    sys.path.append(str(repo_dir))

# Import EMMA module
import EMMA.event_emma as em  
import EMMA.event_pca_error as ep
import EMMA.uncertainty_propagation_t as up
importlib.reload(ep)
importlib.reload(em)
importlib.reload(up)

# Define data directory using relative path logic
data_dir = repo_dir / "Data/GrabSample_data"

# Define output directory using relative path logic
output_dir = repo_dir / "Output/EMMA-uncertainty"

# Load the full RI25 dataset
df = pd.read_csv(data_dir / "RI23-IC-ICP-isotope-toc-joined.csv")

####################
# Hungerford RI23 events #
####################

hungerford_tracers = ['Ca_mg_L', 'Cu_mg_L', 'Cl_mg_L', 'K_mg_L', 'Mg_mg_L', 'dD', 'd18O', 'Na_mg_L']

# 1. Run EMMA function
(
    hungerford_fmelt_fractions_df,
    hungerford_fmelt_scaler,
    hungerford_fmelt_pca,
    hungerford_fmelt_endmembers_df,
) = em.run_emma_event(
    data=df,
    site="Hungerford",
    start_date="2023-03-30 00:00:00",
    end_date="2023-04-04 00:00:00",
    endmember_ids=[
                   "RI23-1001", # Baseflow 02/09/2023
                   #"RI23-1035", # 03/22/2023 Baseflow
                   #"RI23-1085", # Baseflow 03/31/23
                   #"RI23-5008", # Groundwater (Mark's well) 03/16/2023
                   #"RI20-HB-DGW-Mark-1" # Mark's well 2020
                   #"RI22-0858", # Mark's well 03/17/2022
                   #"RI22-0864", # Mark's well 07/27/2022
                   #"RI23-5014", # Soil water lysimeter dry 04/12/23
                   "RI23-5013", # Soil water lysimeter wet 04/12/23
                   "RI23-5017", # Snowmelt lysimeter 04/12/23
                   "RI23-1061" # Snowmelt lysimeter 03/28/23
                    ],
)

# 2. Extract raw endmember rows matching these IDs (to calculate endmember SDs)
em_raw_subset = df[df["Sample ID"].isin(["RI23-1001", "RI23-5013", "RI23-5017", "RI23-1061"])]

# 3. Define streamwater samples for the event window
stream_event_df = df[
    (df["Site"] == "Hungerford")
    & (df["Type"].isin(["Grab", "Grab/Isco", "Baseflow", "Isco"]))
    & (pd.to_datetime(df["Date"] + " " + df["Time"]) >= pd.to_datetime("2023-03-30 00:00:00"))
    & (pd.to_datetime(df["Date"] + " " + df["Time"]) <= pd.to_datetime("2023-04-04 00:00:00"))
].copy()
stream_event_df["Datetime"] = pd.to_datetime(
    stream_event_df["Date"] + " " + stream_event_df["Time"]
)

# 4. Define tracer analytical uncertainties (Optional: edit these to match lab detection limits)
analytical_sd = {
    "Ca_mg_L": 0.05,  # mg/L
    "Cl_mg_L": 0.15,  # mg/L
    "Si_mg_L": 0.10,  # mg/L
    "K_mg_L": 0.10,  # mg/L
    "Cu_mg_L": 0.10,  # mg/L
    "Mg_mg_L": 0.02,  # mg/L
    "dD": 0.8,  # per mil (stable isotope precision)
    "d18O": 0.08,  # per mil
    "Na_mg_L": 0.05,  # mg/L
}

# 5. Calculate Genereux Uncertainties
uncertainty_df = up.propagate_genereux_uncertainty(
    stream_df=stream_event_df,
    em_grouped=hungerford_fmelt_endmembers_df,
    em_raw=em_raw_subset,
    tracers=hungerford_tracers,
    analytical_sd=analytical_sd,
    confidence_level=0.7
)

# 6. Merge fractions and their calculated uncertainties for a complete table
results_with_error = pd.merge(
    hungerford_fmelt_fractions_df, uncertainty_df, on=["Sample ID", "Datetime"]
)

# Identify which uncertainty columns were actually generated
uncertainty_cols = [
    col for col in results_with_error.columns if "Uncertainty_1sig" in col
]
# Extract the base fraction names (e.g., "Groundwater", "Snowmelt lysimeter")
fraction_cols = [col.replace("_Uncertainty_1sig", "") for col in uncertainty_cols]

# Combine them in alternating order: [Fraction_1, Uncertainty_1, Fraction_2, Uncertainty_2...]
display_cols = ["Sample ID", "Datetime"]
for frac, unc in zip(fraction_cols, uncertainty_cols):
    display_cols.extend([frac, unc])

# Print the head of the dynamically built column list
print(results_with_error[display_cols].head())
results_with_error.head(20)

   Sample ID            Datetime
0  RI23-1085 2023-03-31 14:00:00
1  RI23-1086 2023-03-31 20:00:00
2  RI23-1087 2023-04-01 02:00:00
3  RI23-1088 2023-04-01 08:00:00
4  RI23-1089 2023-04-01 14:00:00


,Sample ID,Datetime,Site,Baseflow,Snowmelt lysimeter,Soil water lysimeter,Sum_Fractions,Baseflow_Uncertainty_70sig,Snowmelt lysimeter_Uncertainty_70sig,Soil water lysimeter_Uncertainty_70sig
0,RI23-1085,2023-03-31 14:00:00,Hungerford,0.684559,6.364304e-02,0.251798,1.0,0.103390,5.945170e-09,0.103385
1,RI23-1086,2023-03-31 20:00:00,Hungerford,0.684772,1.029019e-02,0.304938,1.0,0.110414,7.609823e-01,0.709358
2,RI23-1087,2023-04-01 02:00:00,Hungerford,0.686955,1.487002e-13,0.313045,1.0,0.111015,7.676304e-01,0.715253
3,RI23-1088,2023-04-01 08:00:00,Hungerford,0.703818,8.970253e-02,0.206480,1.0,0.103478,3.012869e-08,0.103478
4,RI23-1089,2023-04-01 14:00:00,Hungerford,0.721186,2.123024e-01,0.066512,1.0,0.106882,6.883633e-01,0.640411
5,RI23-1091,2023-04-02 02:00:00,Hungerford,NaN,NaN,NaN,0.0,0.084970,5.635838e-01,0.523408
